# Bab 23: Analisis Deret Waktu dan Peramalan

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab23/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab23")
print("Folder kerja:", os.getcwd())

## Listing 23.1: Pola mingguan dan dekomposisi STL

`kode/bab23/komponen.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab23')
import sys; sys.path.insert(0, "kode/bab23")
import numpy as np
from statsmodels.tsa.seasonal import STL
from siapkan import harian, mingguan

h = harian()
hari = (h.biji / h.gerai)[h.tutup == 0]
pola = hari.groupby(hari.index.dayofweek).mean()
print("biji per gerai per hari menurut hari (Senin=0):")
print("  " + "  ".join(f"{v:.2f}" for v in pola))

w = mingguan()
stl = STL(w.per_gerai, period=52, robust=True).fit()
kuat_tren = 1 - stl.resid.var() / (stl.trend + stl.resid).var()
kuat_mus = 1 - stl.resid.var() / (stl.seasonal + stl.resid).var()
print(f"\nSTL mingguan: kekuatan tren {kuat_tren:.2f}, "
      f"kekuatan musiman {kuat_mus:.2f}")
tren = stl.trend.resample("YE").mean()
print("tren rata-rata per tahun:",
      ", ".join(f"{t.year} {v:.2f}" for t, v in tren.items()))
besar = stl.resid.abs().nlargest(4)
print("sisaan terbesar (minggu berakhir):")
for t, v in besar.items():
    print(f"  {t.date()}  {stl.resid[t]:+.2f}  ramadan {w.ramadan[t]}"
          f" tutup {w.tutup[t]} pasca {w.pasca[t]}")
with open("gambar/bab23/stl.dat", "w") as f:
    f.write("t y tren mus sisa\n")
    for i, t in enumerate(w.index):
        f.write(f"{t.year + (t.dayofyear - 1) / 365:.3f} "
                f"{w.per_gerai[t]:.3f} {stl.trend[t]:.3f} "
                f"{stl.seasonal[t]:.3f} {stl.resid[t]:.3f}\n")

## Listing 23.2: Autokorelasi dan ramalan AR(1) mini

`kode/bab23/acf_mini.py`

In [ ]:
import numpy as np
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.stattools import acf, pacf

x = np.array([6.0, 6.4, 6.2, 6.8, 7.0, 6.6, 7.2, 7.4])
d = x - x.mean()
r = [(d[k:] * d[:len(d) - k]).sum() / (d ** 2).sum() for k in (1, 2)]
print(f"rata-rata {x.mean():.2f}; r1 = {r[0]:.3f}, r2 = {r[1]:.3f}")
print("statsmodels acf:", acf(x, nlags=2).round(3))

mu, phi = x.mean(), 0.6
ramal, akhir = [], x[-1]
for h in range(1, 4):
    akhir = mu + phi * (akhir - mu)
    ramal.append(round(float(akhir), 3))
print("ramalan AR(1) 3 langkah:", ramal)

print("\nACF dan PACF teoretis (lag 1-4):")
for nama, ar, ma in [("AR(1) phi=0,7", [1, -.7], [1]),
                     ("MA(1) theta=0,7", [1], [1, .7]),
                     ("AR(2) 0,5; 0,3", [1, -.5, -.3], [1])]:
    p = ArmaProcess(ar, ma)
    print(f"  {nama:16s} ACF  {p.acf(5)[1:].round(2)}")
    print(f"  {'':16s} PACF {p.pacf(5)[1:].round(2) + 0.0}")

## Listing 23.3: Uji stasioneritas dan identifikasi ARIMA

`kode/bab23/identifikasi.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab23')
import sys; sys.path.insert(0, "kode/bab23")
import warnings
import numpy as np
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf, adfuller, pacf
from siapkan import mingguan
warnings.simplefilter("ignore")

w = mingguan()
y = w.per_gerai[:"2025-12-28"]
for nama, s in [("level", y), ("selisih pertama", y.diff().dropna())]:
    stat, p = adfuller(s, autolag="AIC")[:2]
    print(f"ADF {nama:16s}: statistik {stat:6.2f}, nilai-p {p:.3f}")

dy = y.diff().dropna()
a, pa = acf(dy, nlags=8), pacf(dy, nlags=8)
batas = 1.96 / np.sqrt(len(dy))
print(f"\nACF/PACF selisih pertama (batas +-{batas:.2f}):")
print("lag   " + " ".join(f"{k:6d}" for k in range(1, 9)))
print("ACF   " + " ".join(f"{v:6.2f}" for v in a[1:]))
print("PACF  " + " ".join(f"{v:6.2f}" for v in pa[1:]))

print("\nkandidat ARIMA(p,1,q), diurutkan menurut AIC:")
hasil = []
for p in range(3):
    for q in range(3):
        m = ARIMA(y, order=(p, 1, q)).fit()
        hasil.append((m.aic, p, q, m))
for aic, p, q, m in sorted(hasil, key=lambda r: r[0])[:4]:
    lb = acorr_ljungbox(m.resid[1:], lags=[10]).lb_pvalue.iloc[0]
    print(f"  ARIMA({p},1,{q})  AIC {aic:7.1f}  Ljung-Box p {lb:.3f}")
with open("gambar/bab23/acf.dat", "w") as f:
    f.write("lag acf pacf\n")
    a, pa = acf(dy, nlags=20), pacf(dy, nlags=20)
    for k in range(1, 21):
        f.write(f"{k} {a[k]:.3f} {pa[k]:.3f}\n")

## Listing 23.4: Studi kasus ramalan kebutuhan biji

`kode/bab23/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab23')
import sys; sys.path.insert(0, "kode/bab23")
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from siapkan import mingguan
warnings.simplefilter("ignore")

w = mingguan()
t = np.arange(len(w))
for k in (1, 2):
    w[f"sin{k}"] = np.sin(2 * np.pi * k * t / 52.18)
    w[f"cos{k}"] = np.cos(2 * np.pi * k * t / 52.18)
FOURIER = ["sin1", "cos1", "sin2", "cos2"]
KALENDER = ["ramadan", "tutup", "libur", "pasca"]

def ramal(latih, uji):
    y = latih.per_gerai
    h = len(uji)
    out = {"naif musiman": w.per_gerai.shift(52).loc[uji.index]}
    hw = ExponentialSmoothing(y, trend="add", damped_trend=True,
                              seasonal="add",
                              seasonal_periods=52).fit()
    out["Holt-Winters"] = pd.Series(hw.forecast(h).to_numpy(),
                                    uji.index)
    for nama, X in [("ARIMA + Fourier", FOURIER),
                    ("ARIMA + Fourier + kalender", FOURIER + KALENDER)]:
        m = SARIMAX(y, exog=latih[X], order=(1, 1, 1))
        m = m.fit(disp=False)
        f = m.get_forecast(h, exog=uji[X])
        out[nama] = pd.Series(f.predicted_mean.to_numpy(), uji.index)
        if "kalender" in nama:
            pi = {a: f.conf_int(alpha=a).to_numpy()
                  for a in (.2, .05)}
            koef = m.params[KALENDER]
    return out, pi, koef

UJI = {"Ramadan 2025": ("2024-12-29", "2025-04-13"),
       "Ramadan 2026": ("2025-12-28", "2026-03-29")}
for judul, (akhir, ujung) in UJI.items():
    latih = w[:akhir]
    uji = w[pd.Timestamp(akhir) + pd.Timedelta(days=7):ujung]
    out, pi, koef = ramal(latih, uji)
    skala = latih.per_gerai.diff().abs().mean()        # MASE
    print(f"\n{judul}: latih s.d. {akhir}, uji {len(uji)} minggu")
    print(f"  {'model':28s} {'MAE':>6s} {'MAPE':>6s} {'MASE':>5s}"
          f" {'MAE Ramadan':>11s}")
    ram = (uji.ramadan > 0) | (uji.tutup > 0)
    for nama, f in out.items():
        e = (uji.per_gerai - f).abs()
        print(f"  {nama:28s} {e.mean():6.3f} "
              f"{(e / uji.per_gerai).mean():6.1%} "
              f"{e.mean() / skala:5.2f} {e[ram].mean():11.3f}")
    for a, (nm) in [(.2, "80%"), (.05, "95%")]:
        lo, hi = pi[a][:, 0], pi[a][:, 1]
        cak = ((uji.per_gerai >= lo) & (uji.per_gerai <= hi)).mean()
        print(f"  cakupan selang {nm}: {cak:.0%}")
print("\nkoefisien kalender (kg per gerai per minggu, per hari):")
print("  " + ", ".join(f"{k} {v:+.3f}" for k, v in koef.items()))

f = out["ARIMA + Fourier + kalender"]
total = pd.DataFrame({"aktual": uji.biji, "ramalan": f * uji.gerai,
                      "atas90": pi[.2][:, 1] * uji.gerai})
print("\nkebutuhan total 2026 (kg), contoh minggu Ramadan:")
print(total.loc["2026-02-15":"2026-03-29"].round(0).to_string())
with open("gambar/bab23/ramalan.dat", "w") as fh:
    fh.write("t y f lo80 hi80 lo95 hi95 naif\n")
    for i, d in enumerate(uji.index):
        fh.write(f"{i + 1} {uji.per_gerai[d]:.3f} "
                 f"{f[d]:.3f} {pi[.2][i, 0]:.3f} {pi[.2][i, 1]:.3f} "
                 f"{pi[.05][i, 0]:.3f} {pi[.05][i, 1]:.3f} "
                 f"{out['naif musiman'][d]:.3f}\n")
with open("gambar/bab23/latih.dat", "w") as fh:
    fh.write("t y\n")
    for d, v in w.per_gerai["2025-06-01":"2025-12-28"].items():
        fh.write(f"{(d - uji.index[0]).days // 7 + 1} {v:.3f}\n")